# Notebook for å autentisere seg mot skyporten og hente ut data fra share

In [ ]:
! pip install -r requirements.txt

In [ ]:
import os
import json
import subprocess
import delta_sharing

from src.auth import generate_access_token
from src.utils import (
    is_auth_valid,
    is_config_expired,
    fetch_delta_sharing_config,
)

In [ ]:
project_id = ""
project_num = ""
provider_full_identifier = f"projects/{project_num}/locations/global/workloadIdentityPools/skyporten-bi-prod/providers/skyporten-bi-provider-prod"
random_id = ""
region = ""
schema_name = "matrikkel_silver_v1_ext"
share_name = f"{random_id}-prod"
file_name = f"{share_name}.json"


CREDENTIALS_PATH = "credentials.json"
CONFIG_PATH = "configs/config.json"
TOKEN_PATH = "tmp_maskinporten_token.txt"
AUTH_JSON_PATH = "authentication.json"
SOURCE_PATH = "share/config.share" #filen du leser fra dersom du har en gyldig config.share, eller ønsker å skrive til dersom share har utløpt

In [ ]:
with open(CONFIG_PATH, 'r') as file:
    config = json.load(file)

token = generate_access_token(
    kid=config.get('kid'),
    scope=config.get('scope'),
    certname=config.get('certname'),
    audience=config.get('audience'),
    client_id=config.get('client_id'),
    token_url=config.get('url'),
)

with open(TOKEN_PATH, 'w') as file:
    file.write(token.get("access_token", ""))



In [ ]:

subprocess.run([
    "gcloud", "iam", "workload-identity-pools", "create-cred-config",
    provider_full_identifier,
    "--credential-source-type=text",
    "--credential-source-file=tmp_maskinporten_token.txt",
    f"--output-file={CREDENTIALS_PATH}"
], check=True)



In [ ]:
os.makedirs("share", exist_ok=True)

if is_config_expired(SOURCE_PATH):
    subprocess.run(["gcloud", "auth", "login", "--cred-file", CREDENTIALS_PATH, "--quiet"], check=True)
    bucket_id = f"sp-{project_id}-{random_id}"
    gcs_file_path = f"gs://{bucket_id}/{file_name}"
    subprocess.run(["gcloud", "storage", "cp", gcs_file_path, SOURCE_PATH], check=True)



In [ ]:
sharing_client = delta_sharing.SharingClient(SOURCE_PATH)

tables = []
try:
    tables = sharing_client.list_all_tables()
except:
    print("Sharen har ikke tilgang til noen tabeller")
    exit(1)

for table in tables:
    print(table.name)

In [ ]:
table_name = tables[0].name #henter første tabell som er tilgjengelig i share, erstatt med ønsket tabellnavn 
table_url = f"{SOURCE_PATH}#{share_name}.{schema_name}.{table_name}"
data = delta_sharing.load_as_pandas(table_url)
print(data.head())

In [ ]:
table_url = f"{SOURCE_PATH}#{share_name}.{schema_name}.{table_name}"
metadata = delta_sharing.get_table_metadata(table_url)

print(metadata)